# Overcomplete qLDLC Basis Decoding

This notebook shows the small moving parts needed to decode with an overcomplete qLDLC check matrix. We load one generated qLDLC code, append independent logical rows to the reduced lattice checks so belief propagation sees redundant constraints, and then map the BP estimate back through the reduced basis for the final correction.

Load the experiment helpers and fix the random seed. The helper resolves generated qLDLC codes from the repository, so the notebook does not depend on an absolute path.

In [1]:
using LinearAlgebra
using Random

function find_repo_file(parts...)
    starts = unique(normpath.([@__DIR__, pwd()]))

    for start in starts
        current = start
        while true
            candidate = normpath(joinpath(current, parts...))
            isfile(candidate) && return candidate

            parent = dirname(current)
            parent == current && break
            current = parent
        end
    end

    error("Could not find repository file: " * joinpath(parts...))
end

include(find_repo_file("experiments", "decode_qldlc_codes.jl"));

Random.seed!(1);

function is_not_logical_error(logical_check, residual; atol = 1e-5)
    logical_coordinates = logical_check' * residual
    return all(abs(x - round(x)) < atol for x in logical_coordinates)
end;

Choose one generated qLDLC code. The generated code contains the reduced classical lattice generator and rational logical vectors. We use `XL` and `ZL` as the two independent logical rows for this walkthrough; `YL` is not needed because it is redundant for the extra-check demonstration.

In [2]:
code_name = "reduced_ldlc_gkp_n_15_5"
code_path = qldlc_generated_code_path(code_name)
code = load_qldlc_code(code_path)

Dict{String, Any} with 9 entries:
  "qubit_generator_lll" => Rational{BigInt}[6443738//90254681 -19390521//902546…
  "dY"                  => 1.49441
  "XL"                  => Rational{BigInt}[-225146076//992801491, 155525996//9…
  "dZ"                  => 1.47456
  "dX"                  => 1.506
  "YL"                  => Rational{BigInt}[103152077//992801491, -127283238//9…
  "generator"           => Rational{BigInt}[-1 0 … 0 0; 0 0 … 0 0; … ; -1896792…
  "ZL"                  => Rational{BigInt}[114673566//992801491, -2919107//992…
  "classical_generator" => [0 0 … 0 0; 0 0 … 0 0; … ; 0 0 … 0 0; 0 0 … 0 0]

Build the reduced and overcomplete lattice generators. The reduced matrix already has one square basis of checks. Appending `XL` and `ZL` as rows makes the matrix overcomplete: it has more checks than coordinates, while still describing the same ambient displacement space.

In [3]:
M_classical = Float64.(Matrix(code["classical_generator"]));
M_reduced = Float64.(Matrix(code["qubit_generator_lll"]));
logical_rows = Float64.(vcat(code["XL"]', code["ZL"]'));

M_overcomplete = vcat(M_reduced, 2 .*  logical_rows);

Convert lattice generators into BP check matrices using the qLDLC convention from the generated-code example, `H = -M * J`. BP runs on the overcomplete check matrix, but the final integer decision and correction use the reduced square matrix and its generator `G`.

In [4]:
n_modes = div(size(M_reduced, 2), 2)
J = symplectic_form(n_modes);

H_reduced = -M_reduced * J;
H_overcomplete = -M_overcomplete * J;

G = J * inv(M_reduced);
logical_check = inv(M_reduced);


Run one smoke decode. The received vector has one coordinate per displacement variable, while the overcomplete Tanner graph has extra check nodes from `XL` and `ZL`. After BP, `hard_decision` is evaluated against `H_reduced` so the correction lives in the reduced basis generated by `G`. A small local-search post-processing step then mutates a copy of that integer decision around the least reliable reduced-basis coordinates.

In [ ]:
noise_std = 0.35 / sqrt(2 * pi)
max_iter = 25
decoder = "lsd"

error_vector = sample_error(noise_std, size(H_overcomplete, 2));
received = copy(error_vector);

tanner_graph = initialize_tanner_graph(H_overcomplete);
ldlc_decoder = LDLCDecoder(
    tanner_graph;
    schedule = :serial,
    algorithm = decoder,
    sigma = noise_std,
    max_iterations = max_iter,
)
bp_estimate = run_decoder!(ldlc_decoder, received);

bp_integer_correction = hard_decision(bp_estimate, H_reduced);
bp_correction = received - G * bp_integer_correction;
residual = error_vector - bp_correction;



println("BP correct: ", is_not_logical_error(logical_check, residual))
println("BP + X correct: ", is_not_logical_error(logical_check, residual + logical_rows[1, :]))
println("BP + Z correct: ", is_not_logical_error(logical_check, residual + logical_rows[2, :]))
println("BP + Y correct: ", is_not_logical_error(logical_check, residual + logical_rows[1, :] + logical_rows[2, :]))


any_correct = is_not_logical_error(logical_check, residual) ||
    is_not_logical_error(logical_check, residual + logical_rows[1, :]) ||
    is_not_logical_error(logical_check, residual + logical_rows[2, :]) ||
    is_not_logical_error(logical_check, residual + logical_rows[1, :] + logical_rows[2, :])

println("Any correct: ", any_correct)



The final tuple is the basic sanity check for the example: the BP graph has more checks than variables, the reduced matrix still supplies the square correction basis, and the residual is tested in logical coordinates.

Sweep a few noise values and plot the empirical logical failure rate. This is intentionally tiny, so it is useful as a notebook sanity plot rather than as a threshold estimate.

In [ ]:
using Plots

function overcomplete_logical_failure(noise_scale; samples = 5, max_iter = 10, decoder = "lsd")
    noise_std = noise_scale / sqrt(2 * pi)
    failures = 0

    for _ in 1:samples
        error_vector = sample_error(noise_std, size(H_reduced, 2))
        received = copy(error_vector)

        tanner_graph = initialize_tanner_graph(H_overcomplete)
        ldlc_decoder = LDLCDecoder(
            tanner_graph;
            schedule = :parallel,
            algorithm = decoder,
            sigma = noise_std,
            max_iterations = max_iter,
        )
        bp_estimate = run_decoder!(ldlc_decoder, received)

        decoded_integer_correction = hard_decision(bp_estimate, H_reduced)
        correction = received - G * decoded_integer_correction
        residual = error_vector - correction


        failures += is_not_logical_error(logical_check, residual) ? 0 : 1
    end

    return failures / samples
end

noise_scales = 0.55:0.05:0.8
samples_per_point = 100
failure_rates = [
    overcomplete_logical_failure(noise_scale; samples = samples_per_point)
    for noise_scale in noise_scales
]

plot(
    collect(noise_scales),
    failure_rates;
    marker = :circle,
    xlabel = "noise scale",
    ylabel = "logical failure rate",
    label = "overcomplete BP",
)


[4.499999999999995, -3.0000000000000004, 3.4999999999999987, 1.4999999999999976, 2.9999999999999964, 5.0, 2.5, -0.5000000000000034, 0.5000000000000037, -2.5000000000000027, -1.999999999999998, 9.40906802564158e-16, 1.5000000000000058, -3.4999999999999973, -2.4999999999999973, 3.0000000000000018, -1.4096404275526597e-15, -0.49999999999999606, -1.0000000000000007, 0.4999999999999983, -1.0000000000000016, 1.0000000000000027, -0.5000000000000017, 1.5000000000000022, 1.5000000000000029, -1.4999999999999978, 1.0000000000000022, -0.5000000000000043, -0.4999999999999975, -0.9999999999999998]
[-4.999999999999998, 2.0000000000000018, -4.999999999999997, -2.9999999999999902, -3.0000000000000013, 2.999999999999985, 2.0, -1.0000000000000029, 3.999999999999992, -1.5511839370680203e-15, -0.9999999999999941, -5.031491695527515e-15, 9.908511992846756e-17, 2.000000000000001, 2.999999999999999, 1.9999999999999922, 0.9999999999999964, 1.0, 8.26330139513935e-15, 0.9999999999999981, -2.0000000000000004, 0.9